# 2. How do you evaluate a RAG chatbot?

In the malnutrition and TB lessons, evaluation meant comparing predictions with **labels**: who really developed
GAM, who really had TB. Count the hits and misses and you have sensitivity and ROC-AUC.

Our chatbot outputs a paragraph. What is the "label" for a paragraph?

| Option | How | Problem |
|---|---|---|
| **Expert reference answers** | a clinician writes the right answer to each test question; compare | slow and costly to write; one question can have many right answers; and **who checked the reference?** An unvalidated reference just moves the error |
| **Expert review of every answer** | a clinician reads each answer and grades it | the gold standard, but too slow to repeat every time we change a setting |
| **An LLM as judge, no reference** | a second, stronger LLM checks each answer against the question and the passages it came from | fast and repeatable; checked once against people before a real decision |

We use the third one every time we change something, and keep expert review for the final decision.

## Three questions a judge can answer without knowing the right answer

A RAG answer can go wrong at two places: the **search** (wrong passages) or the **model** (right passages, bad
answer). Three checks, known as the **RAG triad**, separate these, and none of them needs a reference answer:

| Check | The judge reads | It asks | A low score means |
|---|---|---|---|
| **Context relevance** | question + retrieved passages | Are these passages about the question? | **search** failed: the model never saw the right page |
| **Groundedness** (also called faithfulness) | answer + retrieved passages | Is what the answer says supported by the passages? | the **model** added something the guideline doesn't say: a hallucination |
| **Answer relevancy** | question + answer | Does the answer address what was asked? | the **model** dodged, drifted or answered a different question |

```
             context relevance                 groundedness
  question ─────────────────────── passages ─────────────────── answer
      └──────────────────────── answer relevancy ─────────────────┘
```

For clinical decision support, **groundedness** matters most: an ungrounded answer is a confident statement the
guideline doesn't support.

What none of the three can tell you: whether the **guideline itself** is right, or whether the answer left out
something important that was in a passage the search didn't find. That still needs people.

**Ragas** is an open-source library that implements these checks: each is a carefully written prompt for the judge,
plus code that turns the judge's replies into a score from 0 to 1.

In [1]:
import os
import pandas as pd

import rag
import rag_eval

pd.set_option("display.width", 200, "display.max_colwidth", 90, "display.max_rows", 40)
assert os.environ.get("OPENROUTER_API_KEY"), "No key yet: open llm-cds/.env, paste the key after OPENROUTER_API_KEY=, save, restart the kernel"
print("assistant:", rag.Settings().model, "  judge:", rag_eval.JUDGE_MODEL)   # the judge: a different, stronger model

assistant: google/gemma-4-26b-a4b-it   judge: anthropic/claude-haiku-5.5


## The test questions

24 questions written for this workshop: 20 that the guidelines answer (single facts, doses, tables, questions
needing two passages, one in Amharic) and 4 they don't (to check that the assistant refuses).

The file also has reference answers. **We don't use them**: nobody has validated them, and the point of this
notebook is to evaluate without them.

In [2]:
questions = rag_eval.load_questions()
print(questions.type.value_counts().to_string(), "\n")
questions

type
answerable      20
out_of_scope     4 



,id,type,question
0,chol_01,answerable,"What findings should make me raise a cholera alert (suspected outbreak) in my area, an..."
1,chol_02,answerable,"A 7-year-old with cholera has stopped vomiting. Which antibiotic should I give, at wha..."
2,chol_03,answerable,A 3-year-old comes to the CTC severely dehydrated from cholera. How should I give the ...
3,chol_04,answerable,"For a reactive oral cholera vaccination campaign with the stockpile vaccines, how far ..."
4,nutr_01,answerable,Our emergency nutrition survey found an acute malnutrition rate of 12% with no aggrava...
5,nutr_02,answerable,"In an emergency vitamin A supplementation round, what preventive dose and how often sh..."
6,nutr_03,answerable,"If a displaced population is relying completely on general food distribution, what is ..."
7,csd_01,answerable,Which diseases are prioritized for climate-sensitive disease surveillance in Ethiopia?
8,csd_02,answerable,"How long does the malaria parasite need to develop inside the mosquito at about 25 °C,..."
9,csd_03,answerable,"In the climate-sensitive disease case definitions, when should a patient be classified..."


## Step 1. What does the search find?

Before any LLM: run only the search (the setup we built in notebook 1: PDF text, 200-word windows, keywords), and
look at which pages come back for each question.

In [3]:
baseline = rag.Settings()                      # docs="pdf", chunking="fixed", retriever="tfidf", k=5
retriever = rag.build(baseline)


def top_sources(question, k=5):
    hits = retriever.search(question, k)
    return "; ".join(rag.source_ids(hits))


questions.assign(sources=questions.question.map(top_sources))[["id", "question", "sources"]]

,id,question,sources
0,chol_01,"What findings should make me raise a cholera alert (suspected outbreak) in my area, an...",ephi_cholera_2022:p14; ephi_cholera_2022:p71; ephi_cholera_2022:p16; ephi_cholera_2022...
1,chol_02,"A 7-year-old with cholera has stopped vomiting. Which antibiotic should I give, at wha...",ephi_cholera_2022:p25; ephi_cholera_2022:p38; ephi_cholera_2022:p70; ephi_cholera_2022...
2,chol_03,A 3-year-old comes to the CTC severely dehydrated from cholera. How should I give the ...,ephi_cholera_2022:p28; ephi_cholera_2022:p29; ephi_cholera_2022:p25; ephi_cholera_2022...
3,chol_04,"For a reactive oral cholera vaccination campaign with the stockpile vaccines, how far ...",ephi_cholera_2022:p70; ephi_cholera_2022:p69; ephi_cholera_2022:p69; ephi_cholera_2022...
4,nutr_01,Our emergency nutrition survey found an acute malnutrition rate of 12% with no aggrava...,ephi_emergency_nutrition_2022:p30; ephi_emergency_nutrition_2022:p48; ephi_emergency_n...
5,nutr_02,"In an emergency vitamin A supplementation round, what preventive dose and how often sh...",ephi_emergency_nutrition_2022:p52; ephi_emergency_nutrition_2022:p52; ephi_emergency_n...
6,nutr_03,"If a displaced population is relying completely on general food distribution, what is ...",ephi_emergency_nutrition_2022:p39; ephi_emergency_nutrition_2022:p38; ephi_emergency_n...
7,csd_01,Which diseases are prioritized for climate-sensitive disease surveillance in Ethiopia?,ephi_climate_sensitive_diseases_2022:p55; ephi_climate_sensitive_diseases_2022:p7; eph...
8,csd_02,"How long does the malaria parasite need to develop inside the mosquito at about 25 °C,...",ephi_climate_sensitive_diseases_2022:p42; ephi_climate_sensitive_diseases_2022:p57; ep...
9,csd_03,"In the climate-sensitive disease case definitions, when should a patient be classified...",ephi_climate_sensitive_diseases_2022:p54; ephi_climate_sensitive_diseases_2022:p55; ep...


> **PAUSE (3 min).** Pick three questions. Do the sources look right (the cholera guideline for a cholera
> question, the SAM guideline for a malnutrition question)? Which questions got sources from the wrong guideline?
> What does the search return for the out-of-scope questions?

Looking at 24 rows by eye is fine once. For 7 setups, every time something changes, we need the judge.

## Step 2. One question, judged by hand

Answer one question and run each of the three checks on it, to see what the judge does.

In [4]:
question = questions.question.iloc[0]
result = rag.answer(question, retriever, baseline)
passages = result["contexts"]                   # the texts of the 5 passages the search returned
print(question, "\n")
print(result["answer"])

What findings should make me raise a cholera alert (suspected outbreak) in my area, and how quickly does it have to be reported? 

A cholera alert (suspected cholera outbreak) is defined by finding at least one of the following: A) Two or more people aged 2 years or older with acute watery diarrhea and severe dehydration, or dying from acute watery diarrhea, from the same area within 1 week of one another; or B) One case of acute watery diarrhea [1]. The provided guidelines do not specify how quickly the alert must be reported.


In [5]:
from ragas.metrics.collections import AnswerRelevancy, ContextRelevance, ResponseGroundedness

judge = rag_eval.judge()
checks = {
    "context relevance": (ContextRelevance(llm=judge), dict(user_input=question, retrieved_contexts=passages)),
    "groundedness":      (ResponseGroundedness(llm=judge), dict(response=result["answer"], retrieved_contexts=passages)),
    "answer relevancy":  (AnswerRelevancy(llm=judge, embeddings=rag_eval.embeddings()),
                          dict(user_input=question, response=result["answer"])),
}
for name, (metric, inputs) in checks.items():
    score = await metric.ascore(**inputs)
    print(f"{name:18} {score.value:.2f}")

context relevance  1.00


groundedness       0.75


answer relevancy   0.83


What happens inside, roughly:

- **Context relevance:** the judge rates, twice, whether the passages are relevant to the question (0, 1 or 2);
  the two ratings are averaged and scaled to 0–1.
- **Groundedness:** the judge rates, twice, whether the answer is supported by the passages: fully (2), partly (1)
  or not (0); averaged and scaled to 0–1. An answer with one invented dose among correct ones gets about 0.5.
- **Answer relevancy:** the judge writes the questions that this answer *would* be an answer to; Ragas measures how
  close in meaning they are to the real question (with an embedding model). It also scores 0 if the answer is
  non-committal ("I'm not sure").

## Step 3. All 24 questions

`rag_eval.evaluate_setup` does the same for every question, 8 at a time, and records everything in **MLflow**:
one run for this setup, with its settings and prompt, and one **trace** per question showing the search, the LLM
call and the three scores. About a minute.

In [6]:
base = rag_eval.evaluate_setup(baseline, "1. baseline: PDF text, fixed, keywords")

2026/10/28 20:59:10 INFO mlflow.store.db.utils: Creating initial MLflow database tables...


2026/10/28 20:59:10 INFO mlflow.store.db.utils: Updating database tables


2026/10/28 20:59:11 INFO mlflow.tracking.fluent: Experiment with name 'guideline-assistant' does not exist. Creating a new experiment.


2026/10/28 20:59:12 INFO mlflow.genai.utils.data_validation: Testing model prediction with the first sample in the dataset. To disable this check, set the MLFLOW_GENAI_EVAL_SKIP_TRACE_VALIDATION environment variable to True.


Evaluating:   0%|          | 0/24 [Elapsed: 00:00, Remaining: ?]

...

Evaluating:  88%|████████▊ | 21/24 [Elapsed: 00:23, Remaining: 00:03]

Evaluating:  96%|█████████▌| 23/24 [Elapsed: 00:23, Remaining: 00:01]

Evaluating: 100%|██████████| 24/24 [Elapsed: 00:27, Remaining: 00:00] [predict_fn: 30%, scorers: 70%]

Evaluating: 100%|██████████| 24/24 [Elapsed: 00:27, Remaining: 00:00] [predict_fn: 30%, scorers: 70%]


✨ Evaluation completed.

Metrics and evaluation results are logged to the MLflow run:
  Run name: 1. baseline: PDF text, fixed, keywords
  Run ID: 3c66785cbc8a401ca9b33f66bb130009

To view the detailed evaluation results with sample-wise scores,
open the Traces tab in the Run page in the MLflow UI.

1. baseline: PDF text, fixed, keywords: context_relevance 0.97, groundedness 0.98, answer_relevancy 0.85, refused_in_scope 4, refused_out_of_scope 4   (MLflow run 3c66785c)


Three averages, over the questions it **answered**, and two counts of refusals.

**Why refusals are counted apart.** For "what insulin dose for a child with type 1 diabetes?" the right behaviour is
the refusal. But a refusal says nothing that the passages could support, and doesn't address the question
(answer relevancy 0): the judge would punish honesty. So: refusals to out-of-scope questions should be **4** (all of them), and refusals
to questions the guidelines *do* answer should be **0**; each one is a missed answer.

## Step 4. Read the worst answers

Averages hide the failures. Sort by the scores, then read the question, the answer and the passages, and decide
for yourself whether the judge is right.

In [7]:
scores = base[["id", "type", "refused"] + rag_eval.METRICS].copy()
scores[rag_eval.METRICS] = scores[rag_eval.METRICS].astype(float).round(2)
scores.sort_values(["groundedness", "context_relevance"])

,id,type,refused,context_relevance,groundedness,answer_relevancy
20,oos_01,out_of_scope,True,0.0,0.00,0.52
21,oos_02,out_of_scope,True,0.0,0.00,0.48
22,oos_03,out_of_scope,True,0.0,0.00,0.51
9,csd_03,answerable,True,0.5,0.00,0.54
14,sam_01,answerable,True,0.5,0.00,0.58
15,sam_02,answerable,True,0.5,0.00,0.48
23,oos_04,out_of_scope,True,0.5,0.00,0.51
17,tb_01,answerable,True,1.0,0.00,0.00
18,tb_02,answerable,False,0.5,0.75,0.92
0,chol_01,answerable,False,1.0,1.00,0.83


In [8]:
def explain(table, question_id, chars=600):
    """The question, the answer, the scores, and the start of each passage the answer was based on."""
    row = table.set_index("id").loc[question_id]
    print(row.question, "\n\nANSWER:", row.answer, "\n")
    print("  ".join(f"{m} {row[m]:.2f}" if pd.notna(row[m]) else f"{m} -" for m in rag_eval.METRICS), row.judge_notes)
    for n, (source, text) in enumerate(zip(row.sources.split("; "), row.passages), 1):
        print(f"\n[{n}] {source}: {text[:chars]}...")


worst = scores[~scores.refused].sort_values(["groundedness", "context_relevance"]).id.iloc[0]
explain(base, worst)

For drug-susceptible TB, when do I record a treatment outcome as 'treatment failure' versus 'lost to follow-up'? 

ANSWER: The provided guidelines do not specify the exact criteria for recording "treatment failure" versus "lost to follow-up" for drug-susceptible TB, other than defining "Treatment after failure" as patients declared treatment failure in their most recent course of treatment as per national protocol [3, 4, 5], and "Treatment after loss to follow-up" as patients declared lost to follow-up at the end of their most recent course of TB treatment who are now decided to be treated with a full course of TB treatment [3, 4, 5]. 

context_relevance 0.50  groundedness 0.75  answer_relevancy 0.92 

[1] moh_tb_2017:p63: This includes cases “transferred out” to another treatment unit as well as cases for whom the treatment outcome is unknown to the reporting unit. Moved to MDR-TB TB Patients who were found to have RR-TB or MDR-TB before fifth month of treatment and who were referred 

> **Exercise (10 min), in pairs.** Take the three answers with the lowest scores (any of the three checks).
> For each, decide:
> 1. Was it the **search** (wrong passages: low context relevance) or the **model** (good passages, but low
>    groundedness or answer relevancy)?
> 2. Do you agree with the judge? Read the passages (`explain(base, "csd_03")`, with the question's id) and check.
>
> Are there any **refusals to in-scope questions**? Look at their sources: did the search find the right page?

## Step 5. The same, in the MLflow UI

In a terminal, from `llm-cds/`, environment active:

```bash
mlflow ui            # http://127.0.0.1:5000
```

Experiment **guideline-assistant** → the run **1. baseline...** → the **Traces** (or Evaluations) tab. Click one
question: the trace shows the `search` step (the passages, with their pages and scores), the `llm` step (the exact
messages and the answer), and the three scores with the judge's reasons on the right. The run's **Parameters**
record the full setup (model, prompt version, chunking, k, judge) and a fingerprint of the passages: everything
needed to reproduce this score.

## What these three checks don't cover

> **PAUSE (5 min), discussion.**
> - The judge sees only the passages the search returned. Can it notice an answer that is grounded in those
>   passages but **incomplete**, because the important line was on a page the search missed?
> - Before this assistant reaches a health worker, what would you still want a clinician to review, and how many
>   answers?

The checks tell us whether the assistant answered **from the guideline**. Whether the guideline answer is the
**complete and right** clinical answer still needs a clinician: grade a sample of answers once, and check the
judge agrees with them, before the assistant is used for real decisions.

## Your turn

Add two questions of your own (from the chat page in notebook 1) and evaluate only those:

In [9]:
mine = pd.DataFrame({"id": ["my_1", "my_2"], "type": ["mine", "mine"],
                     "question": ["How should ORS be prepared at home?",
                                  "What are the danger signs in a child with severe acute malnutrition?"]})
my_scores = rag_eval.evaluate_setup(baseline, "my questions", mine)
my_scores[["question", "sources", "answer"] + rag_eval.METRICS]

2026/10/28 20:59:42 INFO mlflow.genai.utils.data_validation: Testing model prediction with the first sample in the dataset. To disable this check, set the MLFLOW_GENAI_EVAL_SKIP_TRACE_VALIDATION environment variable to True.


Evaluating:   0%|          | 0/2 [Elapsed: 00:00, Remaining: ?]

Evaluating:  50%|█████     | 1/2 [Elapsed: 00:04, Remaining: 00:04]

Evaluating: 100%|██████████| 2/2 [Elapsed: 00:09, Remaining: 00:00]

Evaluating: 100%|██████████| 2/2 [Elapsed: 00:09, Remaining: 00:00] [predict_fn: 17%, scorers: 83%]

Evaluating: 100%|██████████| 2/2 [Elapsed: 00:09, Remaining: 00:00] [predict_fn: 17%, scorers: 83%]


✨ Evaluation completed.

Metrics and evaluation results are logged to the MLflow run:
  Run name: my questions
  Run ID: 39268ac4819c4c46a00f319a249135c8

To view the detailed evaluation results with sample-wise scores,
open the Traces tab in the Run page in the MLflow UI.

my questions: context_relevance 1, groundedness 1, answer_relevancy 0.84, refused_in_scope 1, refused_out_of_scope 0   (MLflow run 39268ac4)


,question,sources,answer,context_relevance,groundedness,answer_relevancy
0,How should ORS be prepared at home?,ephi_cholera_2022:p40; ephi_cholera_2022:p40; ephi_cholera_2022:p39; ephi_cholera_2022...,ORS must be prepared with safe water treated with appropriate methods [1]. The volume ...,1.0,1.0,0.838791
1,What are the danger signs in a child with severe acute malnutrition?,ephi_cholera_2022:p36; ephi_emergency_nutrition_2022:p2; ephi_emergency_nutrition_2022...,The guidelines provided do not cover this.,0.5,0.0,0.000000
